# 🔍 Notebook 2: Observabilidad con Langfuse

## Objetivos
- Crear una cuenta gratuita en Langfuse Cloud
- Entender el modelo de datos: **trace → span → generation**
- Instrumentar el agente TechShop con el decorador `@observe`
- Generar trazas y analizarlas en el dashboard
- Identificar métricas clave: latencia, tokens, tool usage

## ¿Por qué observabilidad?
Cuando un LLM falla, no hay stacktrace. La respuesta simplemente "parece rara".
La observabilidad nos permite ver **exactamente** qué pasó dentro del agente:
qué herramientas llamó, qué le devolvieron, cuántos tokens consumió y cuánto tardó.

## Tutorial: Configurar Langfuse Cloud (5 minutos)

### Paso 1: Crear cuenta
1. Ve a **[cloud.langfuse.com](https://cloud.langfuse.com)**
2. Click en **Sign Up** (puedes usar GitHub, Google o email)
3. El plan **Hobby** es gratuito: 50K observaciones/mes, retención 30 días

### Paso 2: Crear proyecto
1. Una vez dentro, click en **New Project**
2. Nombre: `techshop-llmops`
3. Click **Create**

### Paso 3: Obtener API Keys
1. Ve a **Settings** (⚙️) en el menú lateral
2. Click en **API Keys**
3. Click en **Create API Key**
4. Copia los 3 valores:
   - `Public Key` → `LANGFUSE_PUBLIC_KEY`
   - `Secret Key` → `LANGFUSE_SECRET_KEY`
   - `Host` → `https://cloud.langfuse.com`

### Paso 4: Actualizar tu .env
```
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxxxxx
LANGFUSE_SECRET_KEY=sk-lf-xxxxxxxx
LANGFUSE_HOST=https://cloud.langfuse.com
```

> **Importante:** Nunca compartas tus Secret Keys. El archivo `.env` ya está en `.gitignore`.

## 1. Setup

In [1]:
import strands
import langfuse
import boto3

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verificar AWS
assert os.getenv("AWS_ACCESS_KEY_ID"), "❌ Falta AWS_ACCESS_KEY_ID"

# Verificar Langfuse
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "❌ Falta LANGFUSE_PUBLIC_KEY - sigue el tutorial arriba"
assert os.getenv("LANGFUSE_SECRET_KEY"), "❌ Falta LANGFUSE_SECRET_KEY"

print("✅ Variables de entorno cargadas")

✅ Variables de entorno cargadas


## 2. Conectar con Langfuse

Langfuse se inicializa automáticamente leyendo las variables de entorno `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` y `LANGFUSE_HOST`.

In [5]:
from langfuse import Langfuse


# Langfuse lee automáticamente las variables de entorno
langfuse = Langfuse()


# Verificar conexión
try:
    langfuse.auth_check()
    host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL")
    print(f"✅ Conectado a Langfuse: {host}")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("   Verifica LANGFUSE_PUBLIC_KEY y LANGFUSE_SECRET_KEY en tu .env")


✅ Conectado a Langfuse: https://cloud.langfuse.com


## 3. Tu primera traza con `@observe`

El decorador `@observe` de Langfuse es la forma más sencilla de instrumentar código Python.
Cada función decorada crea una **observación** (span) dentro de una **traza** (trace).

```
Traza (trace)
└── Observación (span) ← @observe crea esto
    └── Sub-observación
```

In [6]:
from langfuse import observe
import time


@observe(name="mi_primera_funcion")
def mi_funcion(x: int) -> int:
    """Una función simple instrumentada con Langfuse."""
    time.sleep(0.1)  # Simular trabajo
    return x * 2

# Ejecutar
resultado = mi_funcion(21)
print(f"Resultado: {resultado}")

# IMPORTANTE: en notebooks, hacemos flush manual para enviar las trazas
langfuse.flush()
print("\n→ Ve a Langfuse > Traces y busca 'mi_primera_funcion'")
print("  Verás: timestamp, duración, input (21), output (42)")

Resultado: 42

→ Ve a Langfuse > Traces y busca 'mi_primera_funcion'
  Verás: timestamp, duración, input (21), output (42)


## 4. Recrear el agente TechShop

Ahora recreamos el agente del Notebook 1, pero esta vez lo instrumentaremos con Langfuse.

In [7]:
from techshop_agent import create_agent

agent = create_agent()

print("✅ Agente TechShop importado desde el paquete")

✅ Agente TechShop importado desde el paquete


## 5. Instrumentar con @observe

Creamos una función `process_query` que envuelve la llamada al agente con trazabilidad completa.

Cada llamada genera una **traza** en Langfuse con:
- `user_id`: quién hizo la consulta
- `session_id`: sesión de conversación
- Input/Output del agente
- Duración de la llamada

In [ ]:
# Actualizar metadata de la traza
from langfuse import langfuse_context


@observe(name="techshop_query")
def process_query(user_query: str, user_id: str = "student", session_id: str = "nb02") -> str:
    """Procesa una consulta al agente con trazabilidad Langfuse.

    Langfuse captura automáticamente:
    - Timestamp de inicio y fin
    - Input (user_query) y Output (respuesta)
    - Duración total
    """
    langfuse_context.update_current_trace(
        user_id=user_id,
        session_id=session_id,
        metadata={"source": "notebook_02", "query_length": len(user_query)},
    )

    response = agent(user_query)
    return str(response)

print("✅ Función process_query instrumentada con Langfuse")

ModuleNotFoundError: No module named 'langfuse.decorators'

## 6. Generar trazas

Ejecutamos varias consultas para generar trazas que luego analizaremos en el dashboard.

Fíjate en la **variedad** de consultas: productos, FAQs, fuera de ámbito, ambiguas.

In [ ]:
queries = [
    "¿Qué portátiles tenéis disponibles?",
    "¿Cuál es la política de devoluciones?",
    "Quiero un móvil con buena cámara",
    "¿Hacéis envíos internacionales?",
    "¿Cuál es la capital de Francia?",
    "Necesito unos auriculares con cancelación de ruido",
    "¿Tenéis tablets?",
    "¿Puedo pagar en cuotas?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'=' * 60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print('=' * 60)
    try:
        response = process_query(
            user_query=query,
            user_id="student01",
            session_id="session-nb02-demo",
        )
        # Mostrar solo primeros 200 chars para no saturar el output
        preview = response[:200] + "..." if len(response) > 200 else response
        print(preview)
    except Exception as e:
        print(f"❌ Error: {e}")

# Flush para enviar todas las trazas
langfuse.flush()
print(f"\n\n✅ {len(queries)} trazas enviadas a Langfuse")

## 7. Explorar trazas en el dashboard

Ahora ve a **[cloud.langfuse.com](https://cloud.langfuse.com)** > tu proyecto > **Traces**.

### Qué buscar:

| Elemento | Dónde mirarlo | Qué revela |
|----------|--------------|------------|
| **Traces** | Lista principal | Cada consulta del usuario |
| **Latencia** | Columna Duration | Tiempo total de respuesta |
| **Input/Output** | Click en una traza | Qué preguntó y qué respondió |
| **User ID** | Filtro lateral | Agrupar por usuario |
| **Session ID** | Filtro lateral | Ver conversaciones completas |

### Preguntas para reflexionar:
1. ¿Hay consultas donde el agente NO usó ninguna herramienta?
2. ¿Alguna consulta tardó mucho más que las demás?
3. ¿El agente respondió preguntas fuera de ámbito? ¿Debería haberlo hecho?

## 8. Consultar trazas por API

También podemos analizar las trazas programáticamente sin salir del notebook.

In [ ]:
# Obtener las últimas trazas
traces = langfuse.fetch_traces(limit=10)

print(f"📊 Últimas {len(traces.data)} trazas:\n")
for t in traces.data:
    duration_ms = "N/A"
    if t.latency is not None:
        duration_ms = f"{t.latency * 1000:.0f}ms"

    input_preview = ""
    if t.input:
        input_text = str(t.input)
        input_preview = input_text[:50] + "..." if len(input_text) > 50 else input_text

    print(f"  [{t.name or 'unnamed':20s}] duration={duration_ms:>8s} | user={t.user_id or 'N/A'} | {input_preview}")

## 9. Métricas clave de observabilidad

La observabilidad no es solo ver trazas. Es poder responder preguntas como:

### Las 4 categorías de métricas LLMOps

| Categoría | Qué medimos | Ejemplo |
|-----------|-------------|----------|
| **Operacionales** | Latencia, errores, throughput | P50 latencia = 2.3s |
| **Coste** | Tokens input/output, € por request | $0.003 por consulta |
| **Calidad** | Alucinaciones, relevancia, fidelidad | 15% de queries sin tool call |
| **Uso** | Queries/hora, temas frecuentes, patrones | 60% de queries son sobre productos |

> **En producción real:** Las categorías 1 y 2 se miden con herramientas clásicas (CloudWatch, Datadog). Las categorías 3 y 4 **necesitan LLMOps** — es lo que hace Langfuse.

## Resumen

| Antes (NB1) | Después (NB2) |
|-------------|---------------|
| "El agente respondió algo" | Veo exactamente qué queries recibió |
| "Parece que funciona" | Conozco la latencia de cada llamada |
| "No sé si usó las herramientas" | Veo qué herramientas llamó y qué devolvieron |
| "¿Cuánto cuesta?" | Puedo ver tokens consumidos |

> **Concepto clave:** Del "parece que funciona" al "sé exactamente qué está pasando y en qué porcentaje de requests".

---

## Siguiente: [Notebook 3 - Prompt Management](../day_2/03_prompt_management.ipynb)